# MISSING FEATURE ENGINEERING TOPICS

The following are important feature engineering topics that should be understood before moving to machine learning model building:

1. Feature Selection
2. Feature Importance
3. Data Leakage Prevention
4. Train-Test Preprocessing Separation

---

# 1. FEATURE SELECTION

Feature selection is the process of **selecting the most useful features** for a machine learning model and removing irrelevant, redundant, or unnecessary features.

### Benefits

- Reduces model complexity
- Can reduce overfitting
- Can improve model performance
- Reduces training time
- Reduces memory usage
- Improves model interpretability

---

## Feature Selection Methods

There are three major approaches:

### 1. Filter Methods

Select features using **statistical properties**, without depending heavily on a specific ML model.

Examples:

- Correlation
- Chi-square test
- ANOVA
- Mutual Information

Example:

```text
Feature
   ↓
Statistical Test
   ↓
Useful / Not Useful
````

### 2. Wrapper Methods

Evaluate different **subsets of features** based on model performance.

Example:

* Recursive Feature Elimination (RFE)

Basic idea:

```text
All Features
     ↓
Train Model
     ↓
Evaluate Features
     ↓
Remove Less Useful Features
     ↓
Train Again
```

### 3. Embedded Methods

Feature selection happens **during model training**.

Examples:

* Lasso Regression
* Decision Tree feature importance
* Regularization-based methods

---

# 2. FEATURE IMPORTANCE

Feature importance tells us **how useful a feature is for making predictions in a particular model**.

Example:

```text
Salary       → 0.45
Experience   → 0.30
Age          → 0.15
Department   → 0.10
```

A higher importance score means the model considers that feature more useful according to that model's importance measure.

### Random Forest Example

```python
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    random_state=42
)

model.fit(X_train, y_train)

importance = pd.Series(
    model.feature_importances_,
    index=X_train.columns
)

importance.sort_values(
    ascending=False
)
```

This ranks the features according to the Random Forest's built-in feature importance measure.

---

## Feature Selection vs Feature Importance

### Feature Selection

> **Which features should be kept?**

Example:

```text
Age
Salary
Experience
Random_ID
```

After selection:

```text
Age
Salary
Experience
```

### Feature Importance

> **Which features does the model consider important for prediction?**

Example:

```text
Salary       → 0.45
Experience   → 0.30
Age          → 0.15
Department   → 0.10
```

### Important

> **Feature importance does NOT mean causation.**

If Salary has high feature importance, it does not prove that Salary causes the target outcome.

---

# 3. DATA LEAKAGE PREVENTION

**Data leakage** occurs when information that should not be available during prediction is used during model training.

This can make model performance appear **unrealistically high**.

---

## Example of Target Leakage

Suppose we want to predict whether a customer will default on a loan.

Available prediction features:

```text
Age
Salary
Credit Score
```

Suppose we also use:

```text
Recovery After Default
```

This is leakage because recovery information becomes available **after the default event**.

The model is receiving information from the future.

### Correct Principle

> A feature must only use information that would actually be available **at prediction time**.

---

# PREPROCESSING DATA LEAKAGE

Leakage can also happen during preprocessing.

Example:

Suppose the dataset contains:

```text
Train Data
+
Test Data
```

and we calculate the mean using the entire dataset:

```python
df["Age"].mean()
```

Then use that mean to fill missing values before splitting.

The mean contains information from the test set.

This allows test-set information to influence the training process.

---

# PREVENTING DATA LEAKAGE

Correct workflow:

```text
Raw Dataset
      ↓
Train-Test Split
      ↓
Train Data        Test Data
      ↓               ↓
Fit preprocessing   Transform only
      ↓               ↓
Transform Train    Transform Test
```

### Rules

* Split the data first.
* Fit preprocessing only on training data.
* Apply the learned preprocessing to validation/test data.
* Never use test-set statistics to fit preprocessing.
* Never use future information when creating features.

---

# 4. TRAIN-TEST PREPROCESSING SEPARATION

The dataset should be split **before fitting preprocessing steps**.

### Train-Test Split

```python
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)
```

Now preprocessing is fitted using only `X_train`.

---

# SCALING EXAMPLE

```python
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(
    X_train
)

X_test_scaled = scaler.transform(
    X_test
)
```

### Important

```text
TRAIN
→ fit_transform()
```

```text
TEST
→ transform()
```

Never fit the scaler separately on the test set.

---

# WHY?

The scaler learns parameters from the training data.

For example:

```text
StandardScaler
→ Mean + Standard Deviation

MinMaxScaler
→ Minimum + Maximum

RobustScaler
→ Median + IQR
```

These parameters must be learned **only from training data**.

Otherwise, information from the test set can influence the model development process.

---

# IMPUTATION EXAMPLE

## Wrong Approach ❌

Calculate the mean using the entire dataset before splitting:

```python
df["Age"] = df["Age"].fillna(
    df["Age"].mean()
)

train_test_split(...)
```

The calculated mean includes information from both training and test data.

---

## Correct Approach ✅

First split the data:

```python
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)
```

Calculate the mean using only training data:

```python
age_mean = X_train["Age"].mean()
```

Fill training data:

```python
X_train["Age"] = X_train["Age"].fillna(
    age_mean
)
```

Use the **same training mean** for test data:

```python
X_test["Age"] = X_test["Age"].fillna(
    age_mean
)
```

### Key Principle

```text
Learn preprocessing parameters from TRAIN
                 ↓
Apply the same parameters to TEST
```

---

# 5. PIPELINE

A **Pipeline** combines preprocessing and model training into a single workflow.

It helps ensure that preprocessing is fitted correctly and reduces the risk of accidental data leakage.

### Example

```python
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression())
])

model.fit(
    X_train,
    y_train
)

predictions = model.predict(
    X_test
)
```

The pipeline:

```text
Training Data
     ↓
StandardScaler.fit()
     ↓
Transform
     ↓
Logistic Regression.fit()

Test Data
     ↓
Same fitted StandardScaler
     ↓
Transform
     ↓
Prediction
```

---

# FEATURE ENGINEERING + DATA LEAKAGE

Feature engineering must also follow the train-test separation rule.

Example of leakage:

```text
Predicting next month's sales
        ↓
Feature uses next month's actual sales
        ↓
LEAKAGE ❌
```

The feature contains information that would not be available when making the prediction.

Correct:

```text
Prediction Time
      ↓
Only information available up to this point
      ↓
Create Features
      ↓
Make Prediction
```

---

# KEY RULES 🧠

### Feature Selection

> Keep useful features and remove irrelevant or unnecessary features.

```text
Features
   ↓
Select Useful Features
   ↓
Model
```

### Feature Importance

> Understand which features contribute to the model's predictions.

```text
Features
   ↓
Model
   ↓
Importance Scores
```

### Data Leakage

> Never allow future, test-set, or otherwise unavailable information to influence training.

```text
Future/Test Information
        ↓
     Training
        ↓
       ❌
```

### Train-Test Preprocessing

> Fit preprocessing on **TRAIN only**.

```text
TRAIN → fit + transform

TEST  → transform only
```

---

# 🧠 FINAL MEMORY

```text
Feature Selection
→ Which features should I keep?

Feature Importance
→ Which features does my model consider important?

Data Leakage
→ Am I accidentally giving the model information it should not have?

Preprocessing
→ Learn parameters from TRAIN only.

TRAIN
→ fit_transform()

TEST
→ transform()
```

> **The most important rule: Never let information from the test set or future data influence feature engineering, preprocessing, or model training.**

```
```
